# Vendedores

Quais vendedores concentram avaliações ruins e atrasos? Compara vendedores com pelo menos 10 pedidos e sua distribuição por estado.

In [ ]:
from pathlib import Path

# Procura a raiz a partir da pasta atual; funciona também nas cópias em outputs/notebooks.
RAIZ_PROJETO = next(
    (pasta for pasta in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (pasta / 'executar_pipeline.py').is_file() and (pasta / 'src').is_dir()),
    None
)
if RAIZ_PROJETO is None:
    raise FileNotFoundError('Execute o notebook dentro da pasta do projeto ou de uma subpasta.')

import pandas as pd

pd.set_option('display.max_columns', None)

CAMINHO_PROCESSADOS = RAIZ_PROJETO / 'data' / 'processed'
MINIMO_PEDIDOS_VENDEDOR = 10
TAMANHO_TOP = 100

df_base = pd.read_parquet(
    f'{CAMINHO_PROCESSADOS}/base_analitica_pedidos.parquet'
)
df_itens_pedidos = pd.read_parquet(
    f'{CAMINHO_PROCESSADOS}/itens_pedidos.parquet'
)


In [ ]:
# Mantém uma ocorrência de cada vendedor dentro do pedido.
df_pedido_vendedor = (
    df_itens_pedidos[['order_id', 'seller_id', 'seller_state']]
    .drop_duplicates(subset=['order_id', 'seller_id'])
    .copy()
)

assert df_base['order_id'].is_unique
assert df_pedido_vendedor[['order_id', 'seller_id']].notna().all().all()

# Cada vendedor deve estar associado a no máximo um estado conhecido.
assert (
    df_itens_pedidos.groupby('seller_id')['seller_state']
    .nunique().le(1).all()
)
df_estados_vendedores = (
    df_itens_pedidos.groupby('seller_id', as_index=False)
    .agg(seller_state=('seller_state', 'first'))
)
df_pedido_vendedor = (
    df_pedido_vendedor.drop(columns='seller_state')
    .merge(df_estados_vendedores, on='seller_id', validate='many_to_one')
)

# Calcula a composição no conjunto completo de itens antes dos filtros.
quantidade_vendedores = (
    df_pedido_vendedor.groupby('order_id')['seller_id'].nunique()
)
df_pedido_vendedor['pedido_multivendedor'] = (
    df_pedido_vendedor['order_id'].map(quantidade_vendedores).gt(1)
)

df_vendedores = (
    df_pedido_vendedor
    .merge(
        df_base.loc[
            df_base['pedido_entregue'].eq(True)
            & df_base['review_score'].isin([1, 2, 3, 4, 5]),
            ['order_id', 'review_score', 'is_late']
        ],
        on='order_id',
        how='inner',
        validate='many_to_one'
    )
)

df_vendedores['review_ruim'] = df_vendedores['review_score'].le(2)
df_vendedores['review_neutra'] = df_vendedores['review_score'].eq(3)
df_vendedores['review_boa'] = df_vendedores['review_score'].ge(4)
df_vendedores['seller_state'] = df_vendedores['seller_state'].fillna('sem_estado')

assert not df_vendedores.duplicated(['order_id', 'seller_id']).any()
print('Pedidos avaliados distintos:', df_vendedores['order_id'].nunique())
print('Relações pedido–vendedor:', len(df_vendedores))


In [ ]:
df_ranking_vendedores = (
    df_vendedores.groupby('seller_id', as_index=False)
    .agg(
        seller_state=('seller_state', 'first'),
        quantidade_pedidos=('order_id', 'nunique'),
        nota_media=('review_score', 'mean'),
        reviews_ruins=('review_ruim', 'sum'),
        reviews_neutras=('review_neutra', 'sum'),
        reviews_boas=('review_boa', 'sum'),
        pedidos_com_prazo_conhecido=('is_late', 'count'),
        pct_pedidos_atrasados=('is_late', 'mean'),
        pct_pedidos_multivendedor=('pedido_multivendedor', 'mean')
    )
)

for tipo in ('ruins', 'neutras', 'boas'):
    df_ranking_vendedores[f'pct_reviews_{tipo}'] = (
        df_ranking_vendedores[f'reviews_{tipo}']
        / df_ranking_vendedores['quantidade_pedidos']
    )

df_ranking_vendedores = (
    df_ranking_vendedores
    .loc[lambda dados: dados['quantidade_pedidos'].ge(MINIMO_PEDIDOS_VENDEDOR)]
    .sort_values(
        ['pct_reviews_ruins', 'quantidade_pedidos', 'seller_id'],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

df_ranking_vendedores.insert(
    0, 'posicao', range(1, len(df_ranking_vendedores) + 1)
)

df_ranking_vendedores_atraso = (
    df_ranking_vendedores
        .loc[lambda dados: dados['quantidade_pedidos'].ge(MINIMO_PEDIDOS_VENDEDOR)]
        .sort_values(
            ['pct_pedidos_atrasados', 'pct_reviews_ruins', 'quantidade_pedidos'],
            ascending=[False, False, True]
        )
        .reset_index(drop=True)
)

df_ranking_vendedores_atraso['posicao'] = range(
    1, len(df_ranking_vendedores_atraso) + 1
)

formatacao_vendedores = {
    'nota_media': '{:.2f}',
    'pct_reviews_ruins': '{:.2%}',
    'pct_reviews_neutras': '{:.2%}',
    'pct_reviews_boas': '{:.2%}',
    'pct_pedidos_atrasados': '{:.2%}',
    'pct_pedidos_multivendedor': '{:.2%}'
}

display(
    df_ranking_vendedores.head(20)
    .style.format(formatacao_vendedores, na_rep='Sem informação')
)

display(
    df_ranking_vendedores_atraso.head(20)
    .style.format(formatacao_vendedores, na_rep='Sem informação')
)
print('Vendedores elegíveis:', len(df_ranking_vendedores))


In [ ]:
df_top100_vendedores = (
    df_ranking_vendedores.loc[
        df_ranking_vendedores['reviews_ruins'].gt(0)
    ]
    .head(TAMANHO_TOP)
    .copy()
)

df_elegiveis_por_estado = (
    df_ranking_vendedores.groupby('seller_state', as_index=False)
    .agg(vendedores_elegiveis=('seller_id', 'nunique'))
)
df_top100_por_estado = (
    df_top100_vendedores.groupby('seller_state', as_index=False)
    .agg(vendedores_no_top100=('seller_id', 'nunique'))
)

df_ranking_estados_vendedores = (
    df_elegiveis_por_estado
    .merge(df_top100_por_estado, on='seller_state', how='left', validate='one_to_one')
)
df_ranking_estados_vendedores['vendedores_no_top100'] = (
    df_ranking_estados_vendedores['vendedores_no_top100'].fillna(0).astype(int)
)

total_elegiveis = len(df_ranking_vendedores)
total_top100 = len(df_top100_vendedores)

df_ranking_estados_vendedores['pct_elegiveis_no_top100'] = (
    df_ranking_estados_vendedores['vendedores_no_top100']
    / df_ranking_estados_vendedores['vendedores_elegiveis']
)
df_ranking_estados_vendedores['pct_participacao_top100'] = (
    df_ranking_estados_vendedores['vendedores_no_top100']
    / total_top100 if total_top100 else float('nan')
)
df_ranking_estados_vendedores['pct_participacao_elegiveis'] = (
    df_ranking_estados_vendedores['vendedores_elegiveis']
    / total_elegiveis if total_elegiveis else float('nan')
)

df_ranking_estados_vendedores = (
    df_ranking_estados_vendedores
    .sort_values(
        ['pct_elegiveis_no_top100', 'vendedores_no_top100', 'seller_state'],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)
df_ranking_estados_vendedores.insert(
    0, 'posicao', range(1, len(df_ranking_estados_vendedores) + 1)
)

display(
    df_ranking_estados_vendedores.style.format({
        'pct_elegiveis_no_top100': '{:.2%}',
        'pct_participacao_top100': '{:.2%}',
        'pct_participacao_elegiveis': '{:.2%}'
    }, na_rep='Sem informação')
)
print('Vendedores efetivamente incluídos no top:', total_top100)


In [ ]:
# Confere se avaliações e contagens foram preservadas.
assert df_ranking_vendedores['seller_id'].is_unique
assert (
    df_ranking_vendedores['reviews_ruins']
    + df_ranking_vendedores['reviews_neutras']
    + df_ranking_vendedores['reviews_boas']
).eq(df_ranking_vendedores['quantidade_pedidos']).all()
assert df_ranking_vendedores['pct_reviews_ruins'].between(0, 1).all()
assert df_ranking_estados_vendedores['vendedores_no_top100'].sum() == total_top100
assert df_ranking_estados_vendedores['vendedores_elegiveis'].sum() == total_elegiveis
assert df_ranking_estados_vendedores['pct_elegiveis_no_top100'].between(0, 1).all()

print('Validações concluídas.')


## Gráficos

Montagem dos gráficos da análise.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from io import BytesIO

plt.rcParams['text.parse_math'] = False
PASTA_GRAFICOS = RAIZ_PROJETO / 'outputs' / 'graficos'
graficos_exportar = {}


def registrar_grafico(figura, caminho, dpi=180, bbox_inches='tight'):
    # Captura antes de fechar a figura; grava os arquivos apenas na seção final.
    buffer = BytesIO()
    figura.savefig(buffer, format='png', dpi=dpi, bbox_inches=bbox_inches)
    nome = Path(caminho).name
    if nome in graficos_exportar:
        raise ValueError(f'Nome de gráfico repetido: {nome}')
    graficos_exportar[nome] = buffer.getvalue()


def concluir_grafico(figura, nome, nota):
    figura.text(0.01, 0.01, nota, fontsize=9, color='#555555')
    figura.tight_layout(rect=[0, 0.09, 1, 0.97])
    registrar_grafico(figura, nome)
    plt.show()
    plt.close(figura)


def grafico_situacoes(tabela, grupo, metrica, denominador, titulo, nome, nota):
    # Ausência de grupo permanece NaN: não representa uma taxa observada de zero.
    grupos = tabela[grupo].drop_duplicates().tolist()
    fig, ax = plt.subplots(figsize=(15, max(5, len(grupos) * 0.62 + 2)))
    cores = {'Atrasado': '#C95050', 'No prazo': '#457B9D', 'Sem informação': '#888888'}
    situacoes = [s for s in cores if s in tabela['situacao_entrega'].unique()]
    largura = 0.75 / max(1, len(situacoes))
    for i, situacao in enumerate(situacoes):
        dados = tabela.loc[tabela['situacao_entrega'].eq(situacao)].set_index(grupo).reindex(grupos)
        taxas = dados[metrica].astype(float)
        barras = ax.barh(
            [j + (i - (len(situacoes) - 1) / 2) * largura for j in range(len(grupos))],
            taxas, height=largura, color=cores[situacao], label=situacao
        )
        rotulos = [
            f'{t:.2%}'.replace('.', ',') + f' (n={int(n):,})'.replace(',', '.')
            if pd.notna(t) and pd.notna(n) else ''
            for t, n in zip(taxas, dados[denominador])
        ]
        ax.bar_label(barras, labels=rotulos, padding=3, fontsize=8)
    ax.set_yticks(range(len(grupos)), [str(g).replace('_', ' ') for g in grupos])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.28)
    ax.set_xticks([0, .2, .4, .6, .8, 1])
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.set_xlabel('Proporção')
    ax.set_title(titulo, loc='left', pad=35)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1), ncol=3, frameon=False)
    ax.grid(axis='x', alpha=.2)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    concluir_grafico(fig, nome, nota)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pathlib import Path

# Reutiliza a raiz identificada na célula de leitura dos dados.
PASTA_GRAFICOS = RAIZ_PROJETO / 'outputs' / 'graficos'

plt.rcParams['text.parse_math'] = False

def salvar_grafico(fig, arquivo, nota):
    # Reserva espaço para o recorte e salva a mesma figura exibida no notebook.
    fig.text(0.01, 0.01, nota, fontsize=9, color='#555555')
    fig.tight_layout(rect=[0, 0.09, 1, 1])
    registrar_grafico(fig, PASTA_GRAFICOS / arquivo, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def barras_taxas(dados, grupo, taxa, quantidade, titulo, arquivo, nota, cores='#C95050'):
    # A ordem das linhas recebidas determina a ordem das barras.
    fig, ax = plt.subplots(figsize=(12, max(5, len(dados) * 0.32 + 2)))
    barras = ax.barh(dados[grupo].astype(str), dados[taxa].astype(float), color=cores)
    ax.invert_yaxis()
    rotulos = [
        f'{t:.2%}'.replace('.', ',') + f' (n = {int(n):,})'.replace(',', '.')
        for t, n in zip(dados[taxa], dados[quantidade])
    ]
    ax.bar_label(barras, labels=rotulos, padding=4, fontsize=9)
    ax.set_title(titulo, loc='left', pad=16)
    ax.set_xlabel('Proporção')
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
    # Começa em zero; margem à direita para não cortar os rótulos.
    limite = min(1, max(0.05, float(dados[taxa].max()) * 1.4))
    ax.set_xlim(0, limite * 1.18)
    ax.set_xticks([limite * i / 4 for i in range(5)])
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    salvar_grafico(fig, arquivo, nota)


In [ ]:
barras_taxas(
    df_ranking_vendedores.head(10), 'seller_id', 'pct_reviews_ruins', 'quantidade_pedidos',
    '10 vendedores com maiores taxas de avaliações ruins', '05_vendedores_top10_avaliacoes.png',
    'Pedidos entregues e avaliados; vendedores com pelo menos 10 pedidos no recorte.\n'
    'Avaliações ruins: notas 1 e 2. n = pedidos avaliados do vendedor.'
)
barras_taxas(
    df_ranking_vendedores_atraso.dropna(subset=['pct_pedidos_atrasados']).head(10),
    'seller_id', 'pct_pedidos_atrasados', 'pedidos_com_prazo_conhecido',
    '10 vendedores com maiores taxas de atraso', '05_vendedores_top10_atrasos.png',
    'Pedidos entregues e avaliados; vendedores com pelo menos 10 pedidos no recorte.\n'
    'n = pedidos com prazo conhecido, denominador da taxa de atraso.', cores='#457B9D'
)


In [ ]:
dados = df_ranking_vendedores.dropna(subset=['pct_pedidos_atrasados'])
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(dados['pct_pedidos_atrasados'].astype(float), dados['pct_reviews_ruins'].astype(float),
           s=22, alpha=0.4, color='#2A9D8F', edgecolors='none')
ax.set(xlabel='Proporção de pedidos atrasados', ylabel='Proporção de avaliações ruins',
       xlim=(-0.02, 1), ylim=(-0.02, 1))
ax.set_title('Atrasos e avaliações ruins por vendedor', loc='left', pad=16)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(alpha=0.2)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
salvar_grafico(fig, '05_vendedores_dispersao.png',
    'Vendedores com pelo menos 10 pedidos entregues e avaliados. Atraso considera somente prazo conhecido.\n'
    'Cada ponto representa um vendedor. Relação descritiva, sem atribuição de responsabilidade individual.')


In [ ]:
dados = df_ranking_estados_vendedores
fig, ax = plt.subplots(figsize=(11, 8))
barras = ax.barh(dados['seller_state'], dados['pct_elegiveis_no_top100'], color='#457B9D')
ax.invert_yaxis()
rotulos = [f'{t:.2%}'.replace('.', ',') + f' ({int(k)}/{int(n)} vendedores)'
           for t, k, n in zip(dados['pct_elegiveis_no_top100'], dados['vendedores_no_top100'], dados['vendedores_elegiveis'])]
ax.bar_label(barras, labels=rotulos, padding=4, fontsize=9)
ax.set_title('Participação no top 100 dentro de cada estado', loc='left', pad=16)
ax.set_xlabel('Vendedores no top 100 / vendedores elegíveis do estado')
ax.set_xlim(0, 1.45)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
salvar_grafico(fig, '05_vendedores_estados_top100.png',
    'Top 100 por taxa de avaliações ruins; mínimo de 10 pedidos entregues e avaliados por vendedor.\n'
    'Somente estados com vendedores elegíveis. Grupos pequenos exigem cuidado na interpretação.')


## Exportações

Salva as tabelas em Parquet e CSV e os gráficos em PNG.

In [ ]:
PASTA_PARQUET = RAIZ_PROJETO / 'outputs' / 'tabelas' / 'parquet'
PASTA_CSV = RAIZ_PROJETO / 'outputs' / 'tabelas' / 'csv'
for pasta in (PASTA_PARQUET, PASTA_CSV, PASTA_GRAFICOS):
    pasta.mkdir(parents=True, exist_ok=True)

tabelas_exportar = {
    '05_ranking_vendedores': df_ranking_vendedores,
    '05_ranking_vendedores_atraso': df_ranking_vendedores_atraso,
    '05_ranking_estados_vendedores': df_ranking_estados_vendedores,
}
for nome, original in tabelas_exportar.items():
    tabela = original.reset_index(drop=True).convert_dtypes()
    caminho = PASTA_PARQUET / f'{nome}.parquet'
    tabela.to_parquet(caminho, index=False)
    pd.testing.assert_frame_equal(tabela, pd.read_parquet(caminho), check_exact=True)
    tabela.to_csv(
        PASTA_CSV / f'{nome}.csv', index=False,
        sep=';', decimal=',', encoding='utf-8-sig'
    )
    print(f'Tabela exportada em Parquet e CSV: {nome} ({len(tabela)} linhas)')

for nome, conteudo in graficos_exportar.items():
    (PASTA_GRAFICOS / nome).write_bytes(conteudo)
    print(f'Gráfico exportado: {nome}')
